# 🦴 FracAtlas Medical Vision-Language Model (VLM) Fine-Tuning
### Parameter-Efficient Fine-Tuning (QLoRA 4-bit) for Musculoskeletal Fracture Detection & Med-VQA

This notebook specializes **Qwen2-VL** (2B or 7B) on the **FracAtlas** radiograph dataset using **QLoRA 4-bit** quantization.

- **Compatible Environments**:
  - **Google Colab Free Tier**: NVIDIA Tesla T4 GPU (16 GB VRAM)
  - **Local Workstation / Laptop**: NVIDIA RTX 3050 6GB (Default: 2B model)
- **Artifact Output**: Trained LoRA adapter saved to `models/` (~150 MB).

## 1. Environment Setup & Dependencies

In [ ]:
# Check active GPU
!nvidia-smi

# Install core deep learning packages
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate peft bitsandbytes qwen-vl-utils pillow
print("✅ Dependencies successfully installed!")

## 2. Load Processed Dataset
Load `train.json` and `val.json` generated by `src/dataset_to_vlm.py`.

In [ ]:
import json
import os

train_path = 'data/processed/train.json'
val_path = 'data/processed/val.json'

if os.path.exists(train_path):
    with open(train_path, 'r') as f:
        train_data = json.load(f)
    print(f"Loaded {len(train_data)} training conversation samples.")
    print("Sample sample entry:", json.dumps(train_data[0], indent=2)[:400], "...")
else:
    print("⚠️ Please run `python src/dataset_to_vlm.py` first to generate train.json!")

## 3. Configure 4-bit Quantization (NF4) & Load Base VLM

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Select model: 2B (Local 6GB GPU) or 7B (Colab 16GB T4)
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading base model: {MODEL_ID}...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Base model loaded successfully in 4-bit precision!")

## 4. Inject LoRA Trainable Adapters

In [ ]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Fine-Tuning Execution

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "models/fracatlas_vlm_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

print("Training configuration ready. Execute training loop or save adapter checkpoints.")
# Save trained LoRA adapter weights
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA Adapter saved to: {OUTPUT_DIR}")

## 6. Validation & Radiograph Inference Test

In [ ]:
from PIL import Image

test_image_path = "data/raw/FracAtlas/FracAtlas/images/Fractured/IMG0000019.jpg"
if os.path.exists(test_image_path):
    image = Image.open(test_image_path)
    print(f"Test radiograph loaded: {test_image_path}, Size: {image.size}")
    # Inference query
    prompt = "<image>\nExamine this musculoskeletal radiograph. Describe your diagnostic findings."
    print("Clinical Prompt:", prompt)
else:
    print(f"Test image not found at {test_image_path}")